Link del paper:

https://arxiv.org/pdf/2301.11305

# Pruebas generales de DetectGPT

## DetectGPT fork

In [ ]:
import pandas as pd

In [ ]:
import os
import shutil

import gc
gc.collect()

%cd /content

target_dir = "/content/DetectCodeGPT"

if os.path.exists(target_dir):
    try:
        shutil.rmtree(target_dir, ignore_errors=False)
    except Exception as e:
        !chmod -R 777 /content/DetectCodeGPT
        !rm -rf /content/DetectCodeGPT
else:

print(os.listdir("/content"))

In [ ]:
!git clone https://github.com/mariaortiz0401/DetectCodeGPT
%cd DetectCodeGPT

In [ ]:
!pip install --no-cache-dir -r requirements.txt


In [ ]:
%cd /content/DetectCodeGPT

!mkdir -p /content/DetectCodeGPT/data/CodeSearchNet/python

import json
from datasets import load_dataset


data_train = load_dataset("Fsoft-AIC/the-vault-function", split_set=["train/small"], languages=['python'], streaming=True)
data_test = load_dataset("Fsoft-AIC/the-vault-function", split_set=["test"], languages=['python'], streaming=True)

def save_vault_stream_to_jsonl(dataset_stream, output_path, max_records=10000):
    available_keys = list(dataset_stream.keys())
    if not available_keys:
        return

    chosen_key = available_keys[0]
    count = 0

    with open(output_path, 'w', encoding='utf-8') as f:
        for sample in iter(dataset_stream[chosen_key]):
            code_content = sample.get("code", "")
            docstring_content = sample.get("docstring", "") or sample.get("comment", "")

            if not docstring_content.strip():
                docstring_content = "A python function."


            docstring_tokens = docstring_content.split()

            record = {
                "code": code_content,
                "original_string": code_content,
                "docstring": docstring_content,
                "docstring_tokens": docstring_tokens,
                "language": "python",
                "prompt": docstring_content,
                "solution": code_content
            }
            f.write(json.dumps(record) + '\n')
            count += 1
            if count >= max_records:
                break


save_vault_stream_to_jsonl(data_train, '/content/DetectCodeGPT/data/CodeSearchNet/python/train.jsonl', max_records=10000)
save_vault_stream_to_jsonl(data_test, '/content/DetectCodeGPT/data/CodeSearchNet/python/test.jsonl', max_records=2000)

!cp /content/DetectCodeGPT/data/CodeSearchNet/python/test.jsonl /content/DetectCodeGPT/data/CodeSearchNet/python/valid.jsonl


In [ ]:
%cd /content/DetectCodeGPT/code-generation

!python generate.py --max_num 30

In [ ]:
!head -n 2 /content/DetectCodeGPT/data/CodeSearchNet/python/train.jsonl

In [ ]:


df = pd.read_json('/content/DetectCodeGPT/data/CodeSearchNet/python/train.jsonl', lines=True)

df_ejemplos = df[['prompt', 'code']].head(2)

for i, row in df_ejemplos.iterrows():
    print(f"=== EJEMPLO {i+1} ===")
    print(f"DOCSTRING (PROMPT):\n{row['prompt']}\n")
    print(f"CÓDIGO HUMANO (SOLUTION):\n{row['code']}\n")
    print("-" * 50)

Se tiene que bajar manualmente todo lo que hace tree-sitter para que identifier_tagging sirva. Hay que modificar identifier-tagging, quitando todo lo que sea de Ruby o PHP

In [ ]:
!pip install "tree-sitter>=0.20.0,<0.22.0"

%cd /content/DetectCodeGPT/code-detection

!mkdir -p tree-sitter

%cd tree-sitter
!git clone https://github.com/tree-sitter/tree-sitter-python.git
!git clone https://github.com/tree-sitter/tree-sitter-java.git
!git clone https://github.com/tree-sitter/tree-sitter-go.git
!git clone https://github.com/tree-sitter/tree-sitter-cpp.git
!git clone https://github.com/tree-sitter/tree-sitter-php.git
!git clone https://github.com/tree-sitter/tree-sitter-javascript.git
!git clone https://github.com/tree-sitter/tree-sitter-c-sharp.git
%cd ..

!ls -lh tree-sitter/tree-sitter-python/src/parser.c

!ls -lh tree-sitter/tree-sitter-java/src/parser.c


In [ ]:
%cd /content/DetectCodeGPT/code-detection

!mkdir -p data/CodeSearchNet/python/

!ln -sf /content/DetectCodeGPT/data/CodeSearchNet/python/train.jsonl ./data/CodeSearchNet/python/train.jsonl
!ln -sf /content/DetectCodeGPT/data/CodeSearchNet/python/train.jsonl ./data/CodeSearchNet/python/CodeLlama-7b-hf-10000-tp0.2.jsonl



In [ ]:
%cd /content/DetectCodeGPT/code-detection
!python main.py --language python --dataset TheVault --dataset_key CodeLlama-7b-hf-10000-tp0.2 --max_num 30 --n_samples 30 --n_perturbations 10

%cd /content/DetectCodeGPT

## Ejemplo 1: código generado por chatgpt

In [ ]:
%%writefile /content/DetectCodeGPT/tarea_estudiante.py
def encontrar_factorial(n):
    if n == 0 or n == 1:
        return 1
    else:
        return n * encontrar_factorial(n-1)

print(encontrar_factorial(5))

In [ ]:
%cd /content/DetectCodeGPT
!python predict.py --file tarea_estudiante.py --perturbations 15

## Ejemplo 2: Código de un estudiante

In [ ]:
%%writefile /content/DetectCodeGPT/tarea_estudiante_noIA.py
##################################################
##########   💻  Práctica: Gritar  💻  ##########
##################################################

## 👇 Escriba su código DEBAJO de esta línea 👇 ##


# Recibe la cadena de la entrada con la función 'input'.

cadena = input()

# 1) Convierte las minúsculas en mayúsculas.

cadena_mayus = cadena.upper()


# 2) Reemplaza la letra 'O' por 'OOO'.

cadena_o= cadena_mayus.replace('o', '000')


# 3) Añade los signos de exclamación "!!!!!" al final.

cadena_o =f"{cadena_o}"!!!!!"


# Envía el resultado a la salida con la función 'print'.



PRINT(cadena_o)
## ☝️ Escriba su código ENCIMA de esta línea ☝️ ##

In [ ]:
%cd /content/DetectCodeGPT
!python predict.py --file tarea_estudiante_noIA.py --perturbations 15

# Carga de archivo de interacciones y ajuste de rutas

In [ ]:
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
sub_file_path = '/content/drive/MyDrive/estudio/UNAL/new_project/data/generated/standardize/standardized-submissions-file.csv'


In [ ]:
sub_file = pd.read_csv(sub_file_path)


In [ ]:
sub_file.columns

In [ ]:
import pandas as pd

# Show every row and column fully without truncation
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)


In [ ]:
sub_file[sub_file['platform'] == 'edunext']['archivo_origen']

In [ ]:
import pandas as pd

with pd.option_context('display.max_colwidth', None):
    print(sub_file['archivo_origen'].head())

In [ ]:
sub_file['archive']

In [ ]:
import os
import pandas as pd

def construir_ruta_codigo_con_sufijo(fila):
    try:
        ruta_origen = str(fila['archivo_origen']).strip()
        raw_nombre = str(fila['_id']).strip()

        nombre_limpio = raw_nombre.replace("[", "").replace("]", "").replace("'", "").replace('"', "")
        if not nombre_limpio.endswith('.py'):
            nombre_limpio = f"{nombre_limpio}.py"

        carpeta_metadata = os.path.dirname(ruta_origen)   # .../submissions/open_edx/metadata
        raiz_open_edx = os.path.dirname(carpeta_metadata) # .../submissions/open_edx

        nombre_csv = os.path.basename(ruta_origen)
        nombre_tarea_code = nombre_csv.replace("_metadata.csv", "_code")

        ruta_final = os.path.join(raiz_open_edx, 'code', nombre_tarea_code, nombre_limpio)
        return ruta_final

    except Exception as e:
        return f"Error_Ruta: {str(e)}"

sub_file['ruta_real_py'] = sub_file.apply(construir_ruta_codigo_con_sufijo, axis=1)

print(sub_file['ruta_real_py'].head(1).values[0])

In [ ]:
import pandas as pd

with pd.option_context('display.max_colwidth', None):
    print(sub_file['ruta_real_py'].head())

In [ ]:
import os
import pandas as pd

df_analisis_rutas = sub_file.dropna(subset=['ruta_real_py']).copy()


df_analisis_rutas['existe_en_drive'] = df_analisis_rutas['ruta_real_py'].apply(os.path.exists)

total_registros = len(df_analisis_rutas)
archivos_existentes = df_analisis_rutas['existe_en_drive'].sum()
archivos_faltantes = total_registros - archivos_existentes
porcentaje_faltantes = (archivos_faltantes / total_registros) * 100

print(f"Total de rutas indexadas en el DataFrame : {total_registros:,}")
print(f"Archivos presentes físicamente en Drive   : {archivos_existentes:,}")
print(f"ARCHIVOS FALTANTES EN GOOGLE DRIVE        : {archivos_faltantes:,}")
print(f"Porcentaje de pérdida (Archivos faltantes): {porcentaje_faltantes:.2f}%")

if archivos_faltantes > 0:
    df_erroneos = df_analisis_rutas[df_analisis_rutas['existe_en_drive'] == False]
    for idx, fila in df_erroneos.head(5).iterrows():
        print(f"  • Fila {idx}: {fila['ruta_real_py']}")

# Detección de IA para total de interacciones (archivos py) en Colab

## Predicción para head 5

In [ ]:
import os
import pandas as pd

df_control = sub_file.dropna(subset=['ruta_real_py']).head(5)

for idx, fila in df_control.iterrows():
    ruta_archivo = fila['ruta_real_py']
    nombre = fila['archive']

    print(f"ENTREGA {idx + 1}: {nombre}")

    if os.path.exists(ruta_archivo):
        # Llamamos a tu script de producción pasándole la ruta exacta del Drive
        !python /content/DetectCodeGPT/predict.py --file "{ruta_archivo}" --perturbations 15
    else:
        print(f"Error: El archivo físico no se encontró en Google Drive.\n")

linea del tiempo IA no IA

buscar exportar modelo

predicción para head 5 con más info

In [ ]:
import os
import sys
import math
import torch
import transformers
import pandas as pd
from tqdm import tqdm

sys.path.append('/content/DetectCodeGPT')
from predict import get_log_likelihood, perturb_code_with_t5


device = "cuda" if torch.cuda.is_available() else "cpu"

base_model_name = "Salesforce/codegen-350M-mono"
mask_model_name = "Salesforce/codet5p-770m"

base_tokenizer = transformers.AutoTokenizer.from_pretrained(base_model_name)
base_model = transformers.AutoModelForCausalLM.from_pretrained(
    base_model_name, torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

mask_tokenizer = transformers.AutoTokenizer.from_pretrained(mask_model_name)
mask_model = transformers.AutoModelForSeq2SeqLM.from_pretrained(mask_model_name).to(device)


df_control = sub_file.dropna(subset=['ruta_real_py']).head(5).copy()
registros_finales = []


for idx, fila in tqdm(df_control.iterrows(), total=len(df_control), desc="Procesando lote"):
    ruta_archivo = fila['ruta_real_py']
    nombre_archivo = fila['archive']

    actividad = os.path.basename(os.path.dirname(ruta_archivo))
    ano = fila.get('ano', 2026)
    mes = fila.get('mes', 2)

    if not os.path.exists(ruta_archivo):
        continue

    try:
        with open(ruta_archivo, "r", encoding="utf-8") as f:
            codigo_estudiante = f.read().strip()

        if not codigo_estudiante or len(codigo_estudiante) < 10:
            continue

        ll_original = get_log_likelihood(codigo_estudiante, base_model, base_tokenizer, device)

        perturbed_codes = perturb_code_with_t5(codigo_estudiante, mask_model, mask_tokenizer, device, n_perturbations=15)

        ll_perturbed_sum = 0
        valid_mutations = 0
        for p_code in perturbed_codes:
            ll_val = get_log_likelihood(p_code, base_model, base_tokenizer, device)
            if ll_val is not None:
                ll_perturbed_sum += ll_val
                valid_mutations += 1

        if valid_mutations == 0 or ll_original is None:
            continue

        ll_perturbed_avg = ll_perturbed_sum / valid_mutations
        discrepancy = ll_original - ll_perturbed_avg

        prob_ia = (1 / (1 + math.exp(-10 * (discrepancy - 0.05)))) * 100

        registros_finales.append({
            "archivo": nombre_archivo,
            "actividad": actividad,
            "ano": ano,
            "mes": mes,
            "LL_Original": round(ll_original, 4),
            "LL_Mutados_Avg": round(ll_perturbed_avg, 4),
            "Discrepancia": round(discrepancy, 4),
            "probabilidad_IA": round(prob_ia, 2)
        })

    except Exception as e:
        print(f"Error procesando {nombre_archivo}: {str(e)}")

df_impacto_ia = pd.DataFrame(registros_finales)

df_impacto_ia.head()

## Detección de IA en código para todas las entregas

In [ ]:
import os
import sys
import math
import torch
import transformers
import pandas as pd
from tqdm import tqdm

# ==========================================================
# 1. CONFIGURACIÓN DE RUTAS Y ARCHIVO DE SALIDA
# ==========================================================
csv_salida_drive = "/content/drive/MyDrive/estudio/UNAL/new_project/data/20260212/submissions/open_edx/impacto_ia_tesis.csv"

df_total = sub_file.dropna(subset=['ruta_real_py', 'archive']).copy()

if os.path.exists(csv_salida_drive):
    try:
        df_progreso_previo = pd.read_csv(csv_salida_drive)
        archivos_ya_procesados = set(df_progreso_previo['archivo'].dropna().unique())
        df_pendientes = df_total[~df_total['archive'].isin(archivos_ya_procesados)].copy()
    except Exception as e:
        archivos_ya_procesados = set()
        df_pendientes = df_total.copy()
else:
    archivos_ya_procesados = set()
    df_pendientes = df_total.copy()


if len(df_pendientes) == 0:
    sys.exit()


sys.path.append('/content/DetectCodeGPT')
from predict import get_log_likelihood

device = "cuda" if torch.cuda.is_available() else "cpu"

base_model_name = "Salesforce/codegen-350M-mono"
mask_model_name = "Salesforce/codet5p-770m"

base_tokenizer = transformers.AutoTokenizer.from_pretrained(base_model_name)
base_model = transformers.AutoModelForCausalLM.from_pretrained(
    base_model_name, torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

mask_tokenizer = transformers.AutoTokenizer.from_pretrained(mask_model_name)
mask_model = transformers.AutoModelForSeq2SeqLM.from_pretrained(mask_model_name).to(device)


def perturb_code_PARALELO(text, mask_model, mask_tokenizer, device, n_perturbations=10):
    prompt = f"Perturb open-source python code maintaining logic: {text}"
    inputs = mask_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)

    with torch.no_grad():
        outputs = mask_model.generate(
            **inputs,
            max_length=256,
            do_sample=True,
            temperature=0.7,
            top_p=0.95,
            num_return_sequences=n_perturbations
        )

    perturbed_texts = []
    for output in outputs:
        p_text = mask_tokenizer.decode(output, skip_special_tokens=True).strip()
        if p_text and len(p_text) > 5:
            import re
            p_text = re.sub(r" <extra_id_\d+> ", "", p_text)
            perturbed_texts.append(p_text)
    return perturbed_texts


lote_actual = []
tamano_lote = 100


for idx, (_, fila) in enumerate(tqdm(df_pendientes.iterrows(), total=len(df_pendientes), desc="Procesando")):
    ruta_archivo = fila['ruta_real_py']
    nombre_archivo = fila['archive']

    actividad = os.path.basename(os.path.dirname(ruta_archivo))
    ano = fila.get('ano', 2026)
    mes = fila.get('mes', 2)

    if not os.path.exists(ruta_archivo):
        continue

    try:
        with open(ruta_archivo, "r", encoding="utf-8") as f:
            codigo_estudiante = f.read().strip()

        if not codigo_estudiante or len(codigo_estudiante) < 10:
            continue

        ll_original = get_log_likelihood(codigo_estudiante, base_model, base_tokenizer, device)

        perturbed_codes = perturb_code_PARALELO(codigo_estudiante, mask_model, mask_tokenizer, device, n_perturbations=10)

        ll_perturbed_sum = 0
        valid_mutations = 0
        for p_code in perturbed_codes:
            ll_val = get_log_likelihood(p_code, base_model, base_tokenizer, device)
            if ll_val is not None:
                ll_perturbed_sum += ll_val
                valid_mutations += 1

        if valid_mutations == 0 or ll_original is None:
            continue

        ll_perturbed_avg = ll_perturbed_sum / valid_mutations
        discrepancy = ll_original - ll_perturbed_avg
        prob_ia = (1 / (1 + math.exp(-10 * (discrepancy - 0.05)))) * 100

        lote_actual.append({
            "archivo": nombre_archivo,
            "actividad": actividad,
            "ano": ano,
            "mes": mes,
            "LL_Original": round(ll_original, 4),
            "LL_Mutados_Avg": round(ll_perturbed_avg, 4),
            "discrepancia_cruda": round(discrepancy, 4),
            "probabilidad_IA": round(prob_ia, 2)
        })

        if len(lote_actual) >= tamano_lote:
            df_lote = pd.DataFrame(lote_actual)
            if os.path.exists(csv_salida_drive):
                df_lote.to_csv(csv_salida_drive, mode='a', header=False, index=False)
            else:
                df_lote.to_csv(csv_salida_drive, mode='w', header=True, index=False)
            lote_actual = []

    except Exception as e:
        continue

if len(lote_actual) > 0:
    df_lote = pd.DataFrame(lote_actual)
    if os.path.exists(csv_salida_drive):
        df_lote.to_csv(csv_salida_drive, mode='a', header=False, index=False)
    else:
        df_lote.to_csv(csv_salida_drive, mode='w', header=True, index=False)


## Muestreo estratificado del 10% por cada mes

Se reduce el volumen total a unos 17.500 ejemplos.
Si un mes tenía el doble de entregas que otro, en tu muestra del 10% seguirá teniendo el doble.

In [ ]:
import pandas as pd

df_muestra_tesis = sub_file.dropna(subset=['ruta_real_py', 'archive']).groupby(['ano', 'mes']).sample(frac=0.10, random_state=42).copy()

print(f"Universo original total: {len(sub_file)} entregas.")
print(f"Tamaño de la muestra (10%): {len(df_muestra_tesis)} entregas.")

print("\nDistribución de la muestra por año y mes:")
print(df_muestra_tesis.groupby(['ano', 'mes']).size())

In [ ]:
import os
import sys
import math
import torch
import transformers
import pandas as pd
from tqdm import tqdm



df_base = sub_file.dropna(subset=['ruta_real_py', 'archive']).copy()

df_base['existe_en_drive'] = df_base['ruta_real_py'].apply(os.path.exists)

df_reales = df_base[df_base['existe_en_drive'] == True].copy()

print(f"Archivos totales indexados en metadata: {len(df_base)}")
print(f"Archivos encontrados físicamente en Drive: {len(df_reales)}")

if len(df_reales) == 0:
    sys.exit()

df_total = df_reales.groupby(['ano', 'mes']).sample(frac=0.10, random_state=42).copy()

print(df_total.groupby(['ano', 'mes']).size())


csv_salida_drive = "/content/drive/MyDrive/estudio/UNAL/new_project/data/20260212/submissions/impacto_ia_tesis_estratificado.csv"

if os.path.exists(csv_salida_drive):
    try:
        df_progreso_previo = pd.read_csv(csv_salida_drive)
        archivos_ya_procesados = set(df_progreso_previo['archivo'].dropna().unique())
        df_pendientes = df_total[~df_total['archive'].isin(archivos_ya_procesados)].copy()
    except Exception as e:
        df_pendientes = df_total.copy()
else:
    df_pendientes = df_total.copy()


if len(df_pendientes) == 0:
    sys.exit()


sys.path.append('/content/DetectCodeGPT')
from predict import get_log_likelihood

device = "cuda" if torch.cuda.is_available() else "cpu"

base_model_name = "Salesforce/codegen-350M-mono"
mask_model_name = "Salesforce/codet5p-770m"

base_tokenizer = transformers.AutoTokenizer.from_pretrained(base_model_name)
base_model = transformers.AutoModelForCausalLM.from_pretrained(
    base_model_name, torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

mask_tokenizer = transformers.AutoTokenizer.from_pretrained(mask_model_name)
mask_model = transformers.AutoModelForSeq2SeqLM.from_pretrained(mask_model_name).to(device)

def perturb_code_PARALELO(text, mask_model, mask_tokenizer, device, n_perturbations=10):
    prompt = f"Perturb open-source python code maintaining logic: {text}"
    inputs = mask_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)

    with torch.no_grad():
        outputs = mask_model.generate(
            **inputs,
            max_length=256,
            do_sample=True,
            temperature=0.7,
            top_p=0.95,
            num_return_sequences=n_perturbations
        )

    perturbed_texts = []
    for output in outputs:
        p_text = mask_tokenizer.decode(output, skip_special_tokens=True).strip()
        if p_text and len(p_text) > 5:
            import re
            p_text = re.sub(r" <extra_id_\d+> ", "", p_text)
            perturbed_texts.append(p_text)
    return perturbed_texts


lote_actual = []
tamano_lote = 100


for idx, (_, fila) in enumerate(tqdm(df_pendientes.iterrows(), total=len(df_pendientes), desc="Procesando")):
    ruta_archivo = fila['ruta_real_py']
    nombre_archivo = fila['archive']

    actividad = os.path.basename(os.path.dirname(ruta_archivo))
    ano = fila.get('ano', 2026)
    mes = fila.get('mes', 2)

    try:
        with open(ruta_archivo, "r", encoding="utf-8") as f:
            codigo_estudiante = f.read().strip()

        if not codigo_estudiante or len(codigo_estudiante) < 10:
            continue

        ll_original = get_log_likelihood(codigo_estudiante, base_model, base_tokenizer, device)

        perturbed_codes = perturb_code_PARALELO(codigo_estudiante, mask_model, mask_tokenizer, device, n_perturbations=10)

        ll_perturbed_sum = 0
        valid_mutations = 0
        for p_code in perturbed_codes:
            ll_val = get_log_likelihood(p_code, base_model, base_tokenizer, device)
            if ll_val is not None:
                ll_perturbed_sum += ll_val
                valid_mutations += 1

        if valid_mutations == 0 or ll_original is None:
            continue

        ll_perturbed_avg = ll_perturbed_sum / valid_mutations
        discrepancy = ll_original - ll_perturbed_avg
        prob_ia = (1 / (1 + math.exp(-10 * (discrepancy - 0.05)))) * 100

        lote_actual.append({
            "archivo": nombre_archivo,
            "actividad": actividad,
            "ano": ano,
            "mes": mes,
            "LL_Original": round(ll_original, 4),
            "LL_Mutados_Avg": round(ll_perturbed_avg, 4),
            "discrepancia_cruda": round(discrepancy, 4),
            "probabilidad_IA": round(prob_ia, 2)
        })

        if len(lote_actual) >= tamano_lote:
            df_lote = pd.DataFrame(lote_actual)
            if os.path.exists(csv_salida_drive):
                df_lote.to_csv(csv_salida_drive, mode='a', header=False, index=False)
            else:
                df_lote.to_csv(csv_salida_drive, mode='w', header=True, index=False)
            lote_actual = []

    except Exception as e:
        continue

if len(lote_actual) > 0:
    df_lote = pd.DataFrame(lote_actual)
    if os.path.exists(csv_salida_drive):
        df_lote.to_csv(csv_salida_drive, mode='a', header=False, index=False)
    else:
        df_lote.to_csv(csv_salida_drive, mode='w', header=True, index=False)


La otra opción es elegir una sola entrega por estudiante...

# Análisis de impacto después de correr en servidor con GPU

In [ ]:
import os

In [ ]:
from google.colab import drive
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

drive.mount('/content/drive')

ruta_csv = '/content/drive/MyDrive/estudio/UNAL/new_project/data/generated/IA/impacto_ia_tesis.csv'
df = pd.read_csv(ruta_csv)

print(df.head())

In [ ]:
print(df['actividad'].value_counts())

In [ ]:

imagenes_ia_dir = (
    "/content/drive/MyDrive/estudio/UNAL/new_project/graphics/imagenes_ia/"
)
os.makedirs(imagenes_ia_dir, exist_ok=True)

## Estandarización de tareas

Se estandarizan las tareas para no diferenciar entre edunext y openedx

In [ ]:
import pandas as pd

mapa_estandarizacion = {
    'edunext_a1_saludar_code': 'actividad_1_saludar_code',
    'actividad_1_saludar_code': 'actividad_1_saludar_code',

    'edunext_a2_gritar_code': 'actividad_2_gritar_code',
    'actividad_2_gritar_code': 'actividad_2_gritar_code',

    'edunext_a2_cadenas_de_texto_code': 'actividad_2_cadenas_de_texto_code',
    'actividad_2_cadenas_de_texto_code': 'actividad_2_cadenas_de_texto_code',

    'edunext_a1_variables_code': 'actividad_1_variables_code',
    'actividad_1_variables_code': 'actividad_1_variables_code',

    'edunext_a3_operadores_numericos_code': 'actividad_3_operadores_numericos_code',
    'actividad_3_operadores_numericos_code': 'actividad_3_operadores_numericos_code',

    'actividad_3_dividir_en_grupos_code': 'actividad_3_dividir_en_grupos_code',
    'edunext_a3_dividir_en_grupos_code': 'actividad_3_dividir_en_grupos_code',

    'edunext_a4_ejemplo_1_2_code': 'actividad_4_ejemplo_1_code',
    'edunext_a4_ejemplo_1_code': 'actividad_4_ejemplo_1_code',
    'actividad_4_ejemplo_1_code': 'actividad_4_ejemplo_1_code',

    'edunext_a5_tarea_1_2_code': 'actividad_5_tarea_1_code',
    'edunext_a5_tarea_1_code': 'actividad_5_tarea_1_code',
    'actividad_5_tarea_1_code': 'actividad_5_tarea_1_code',

    'edunext_a6_cursores_y_areas_code': 'actividad_6_cursores_y_areas_code',
    'edunext_a6a_cursores_y_areas_code': 'actividad_6_cursores_y_areas_code',
    'actividad_6_cursores_y_areas_code': 'actividad_6_cursores_y_areas_code',

    'edunext_a7_bisisestos_code': 'actividad_7_anos_bisisestos_code',
    'edunext_a7a_bisisestos_code': 'actividad_7_anos_bisisestos_code',
    'actividad_7_anos_bisisestos_code': 'actividad_7_anos_bisisestos_code',

    'edunext_a8_while_code': 'actividad_8_while_code',
    'actividad_8_while_code': 'actividad_8_while_code',

    'edunext_a9_for_code': 'actividad_9_for_code',
    'actividad_9_for_code': 'actividad_9_for_code',

    'edunext_a10_ejemplo_2_code': 'actividad_10_ejemplo_2_code',
    'actividad_10_ejemplo_2_code': 'actividad_10_ejemplo_2_code',

    'edunext_a11_tarea_2_code': 'actividad_11_tarea_2_code',
    'actividad_11_tarea_2_code': 'actividad_11_tarea_2_code',

    'edunext_a12_trozos_en_cadenas_code': 'actividad_12_trozos_en_cadenas_code',
    'actividad_12_trozos_en_cadenas_code': 'actividad_12_trozos_en_cadenas_code',

    'edunext_a13_listas_code': 'actividad_13_listas_code',
    'actividad_13_listas_code': 'actividad_13_listas_code',

    'edunext_a14_diccionarios_code': 'actividad_14_diccionarios_code',
    'actividad_14_diccionarios_code': 'actividad_14_diccionarios_code',

    'edunext_a15_conjuntos_code': 'actividad_15_conjuntos_code',
    'actividad_15_conjuntos_code': 'actividad_15_conjuntos_code',

    'edunext_a16_ejemplo_3_code': 'actividad_16_ejemplo_3_code',
    'actividad_16_ejemplo_3_code': 'actividad_16_ejemplo_3_code',

    'actividad_17_tarea_3_code': 'actividad_17_tarea_3_code',
    'edunext_a17_tarea_3_code': 'actividad_17_tarea_3_code',

    'edunext_a0_demo_code': 'actividad_0_demo_code',
    'actividad_0_demo_code': 'actividad_0_demo_code'
}

df['actividad_estandar'] = df['actividad'].map(mapa_estandarizacion)

print(df['actividad_estandar'].value_counts())

nulos = df['actividad_estandar'].isna().sum()
if nulos > 0:
    print(f"\n hay {nulos} registros que no coincidieron con ninguna clave del mapa.")
else:
    print("\nEl 100% de los registros fueron homologados.")

## Eliminación de tarea DEMO

Se elimina la tarea demo

In [ ]:
df_filtrado = df[df['actividad_estandar'].str.lower() != 'actividad_0_demo_code'].copy()
print(f"Registros de 'actividad_0_demo_code' removidos.")
print(f"Dataset limpio para análisis: {len(df_filtrado):,} registros.")

In [ ]:
df_filtrado

## Primera gráfica con umbral del 95%

Se realiza una primera gráfica, con un umbral de IA del 95%. Es decir, si la probabilidad_IA es mayor a 95, es clasifica como IA

In [ ]:
UMBRAL_IA = 95.0

df_filtrado['es_IA'] = df_filtrado['probabilidad_IA'] >= UMBRAL_IA if 'UMBRAL_IA' in locals() else df_filtrado['probabilidad_IA'] >= UMBRAL_IA

linea_tiempo_limpia = df_filtrado.groupby('ano').agg(
    total_entregas=('probabilidad_IA', 'count'),
    entregas_IA=('es_IA', 'sum')
).reset_index()

linea_tiempo_limpia['porcentaje_IA'] = (linea_tiempo_limpia['entregas_IA'] / linea_tiempo_limpia['total_entregas']) * 100


import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 6))

ax = sns.lineplot(
    data=linea_tiempo_limpia,
    x='ano',
    y='porcentaje_IA',
    marker='o',
    linewidth=2.5,
    markersize=8,
    color='#2ca02c'
)

##plt.title('Evolución Histórica de Entregas con IA (Excluyendo Actividad Demo)', fontsize=14, pad=15, fontweight='bold')
plt.xlabel('Año académico', fontsize=12, labelpad=10)
plt.ylabel(f'Porcentaje de entregas con probabilidad ≥ {UMBRAL_IA}%', fontsize=12, labelpad=10)

plt.xticks(linea_tiempo_limpia['ano'].unique(), fontsize=11)
plt.yticks(fontsize=11)

for index, fila in linea_tiempo_limpia.iterrows():
    ax.text(
        fila['ano'],
        fila['porcentaje_IA'] + 1.5,
        f"{fila['porcentaje_IA']:.1f}%",
        color='black',
        ha="center",
        fontsize=10,
        fontweight='semibold'
    )

if 2022 in linea_tiempo_limpia['ano'].values:
    plt.axvline(x=2022.9, color='crimson', linestyle='--', alpha=0.7, linewidth=1.5)

plt.ylim(0, max(linea_tiempo_limpia['porcentaje_IA'].max() + 10, 50))
plt.tight_layout()

nombre_archivo = os.path.join(imagenes_ia_dir, "1_grafica_ia_1.pdf")
plt.savefig(nombre_archivo, format="pdf", bbox_inches="tight", pad_inches=0.1)
plt.show()

## Mapa de calor: valores de probabilidad de IA por año

Observamos los valores de IA por tarea, por año

In [ ]:
tabla_diagnostico = df_filtrado.pivot_table(
    index='actividad_estandar',
    columns='ano',
    values='probabilidad_IA',
    aggfunc='mean'
)
num_act = tabla_diagnostico.index.str.extract(r'actividad_(\d+)')[0].astype(int)


orden_correcto = (
    pd.DataFrame({'num': num_act, 'nombre': tabla_diagnostico.index})
    .sort_values(by=['num', 'nombre'])['nombre']
    .tolist()
)

tabla_diagnostico = tabla_diagnostico.reindex(orden_correcto)

pd.set_option('display.float_format', lambda x: '%.2f' % x if not pd.isna(x) else '-')
print(tabla_diagnostico.head(20))


plt.figure(figsize=(14, 10))
años_columnas = [col for col in tabla_diagnostico.columns if col != 'Total_Entregas_Historico']

sns.heatmap(
    tabla_diagnostico[años_columnas],
    annot=True,
    fmt=".1f",
    cmap="YlOrRd",
    linewidths=.5,
    cbar_kws={'label': 'Probabilidad Promedio de IA (%)'}
)

#plt.title('Matriz de Diagnóstico: Probabilidad Promedio de IA por Tarea y Año', fontsize=14, pad=15, fontweight='bold')
plt.xlabel('Año académico', fontsize=12)
plt.ylabel('Nombre de la actividad / aarea', fontsize=12)
plt.tight_layout()

nombre_archivo = os.path.join(imagenes_ia_dir, "2_grafica_ia_2.pdf")
plt.savefig(nombre_archivo, format="pdf", bbox_inches="tight", pad_inches=0.1)
plt.show()

## Eliminación de tareas con valores muy altos (falsos positivos)

Se observa que hay tareas de ejemplo que tienen valores demasiado altos en años previos a chatgpt. Las eliminamos de todos los años.

    'actividad_10_ejemplo_2_code',
    'actividad_11_tarea_2_code',
    'actividad_17_tarea_3_code',
    'actividad_16_ejemplo_3_code',
    'actividad_12_trozos_en_cadenas_code'

Quedamos con 15 tareas

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


tareas_a_excluir = [
    'actividad_10_ejemplo_2_code',
    'actividad_11_tarea_2_code',
    'actividad_17_tarea_3_code',
    'actividad_16_ejemplo_3_code',
    'actividad_12_trozos_en_cadenas_code'
]

df_heatmap = df_filtrado[~df_filtrado['actividad_estandar'].isin(tareas_a_excluir)].copy()

tabla_diagnostico = df_heatmap.pivot_table(
    index='actividad_estandar',
    columns='ano',
    values='probabilidad_IA',
    aggfunc='mean'
)

num_act = tabla_diagnostico.index.str.extract(r'actividad_(\d+)')[0].astype(int)

orden_correcto = (
    pd.DataFrame({'num': num_act, 'nombre': tabla_diagnostico.index})
    .sort_values(by=['num', 'nombre'])['nombre']
    .tolist()
)

tabla_diagnostico = tabla_diagnostico.reindex(orden_correcto)

pd.set_option('display.float_format', lambda x: '%.2f' % x if not pd.isna(x) else '-')
print(tabla_diagnostico.head(20))


plt.figure(figsize=(14, 10))
años_columnas = [col for col in tabla_diagnostico.columns if col != 'Total_Entregas_Historico']

sns.heatmap(
    tabla_diagnostico[años_columnas],
    annot=True,
    fmt=".1f",
    cmap="YlOrRd",
    linewidths=.5,
    cbar_kws={'label': 'Probabilidad Promedio de IA (%)'}
)

#plt.title('Matriz de Diagnóstico: Probabilidad Promedio de IA por Tarea y Año (Tareas Orgánicas)', fontsize=14, pad=15, fontweight='bold')
plt.xlabel('Año académico', fontsize=12)
plt.ylabel('Nombre de la actividad / tarea', fontsize=12)
plt.tight_layout()

# Guardar la nueva versión limpia
nombre_archivo = os.path.join(imagenes_ia_dir, "3_grafica_ia_3.pdf")
plt.savefig(nombre_archivo, format="pdf", bbox_inches="tight", pad_inches=0.1)
plt.show()

## Eliminación del 2021

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


tareas_a_excluir = [
    'actividad_10_ejemplo_2_code',
    'actividad_11_tarea_2_code',
    'actividad_17_tarea_3_code',
    'actividad_16_ejemplo_3_code',
    'actividad_12_trozos_en_cadenas_code'
]

df_heatmap = df_filtrado[~df_filtrado['actividad_estandar'].isin(tareas_a_excluir)].copy()


tabla_diagnostico = df_heatmap.pivot_table(
    index='actividad_estandar',
    columns='ano',
    values='probabilidad_IA',
    aggfunc='mean'
)

num_act = tabla_diagnostico.index.str.extract(r'actividad_(\d+)')[0].astype(int)

orden_correcto = (
    pd.DataFrame({'num': num_act, 'nombre': tabla_diagnostico.index})
    .sort_values(by=['num', 'nombre'])['nombre']
    .tolist()
)

tabla_diagnostico = tabla_diagnostico.reindex(orden_correcto)

columnas_validas = [col for col in tabla_diagnostico.columns if col not in [2021, '2021', 'Total_Entregas_Historico']]
tabla_diagnostico = tabla_diagnostico[columnas_validas]

pd.set_option('display.float_format', lambda x: '%.2f' % x if not pd.isna(x) else '-')
print(tabla_diagnostico.head(20))


plt.figure(figsize=(14, 10))

sns.heatmap(
    tabla_diagnostico,
    annot=True,
    fmt=".1f",
    cmap="YlOrRd",
    linewidths=.5,
    cbar_kws={'label': 'Probabilidad Promedio de IA (%)'}
)

#plt.title('Matriz de Diagnóstico: Probabilidad Promedio de IA por Tarea y Año', fontsize=14, pad=15, fontweight='bold')
plt.xlabel('Año académico', fontsize=12)
plt.ylabel('Nombre de la actividad / tarea', fontsize=12)
plt.tight_layout()

nombre_archivo = os.path.join(imagenes_ia_dir, "4_grafica_ia_4.pdf")
plt.savefig(nombre_archivo, format="pdf", bbox_inches="tight", pad_inches=0.1)

plt.show()


## Propuesta: reducción de falsos positivos

Se realiza una limpieza, como prueba, de tareas basados en los valores que tienen en 2021 y 2022, para eliminar tareas falsos positivos

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from google.colab import drive

UMBRAL_ESTRICTO = 90.0
TOLERANCIA_MAXIMA_CONTROL = 15.0  # maximo % permitido de falsos positivos en años base

tareas_a_excluir = [
    'actividad_10_ejemplo_2_code',
    'actividad_11_tarea_2_code',
    'actividad_17_tarea_3_code',
    'actividad_16_ejemplo_3_code',
    'actividad_12_trozos_en_cadenas_code'
]

df_filtrado = df_filtrado[~df_filtrado['actividad_estandar'].isin(tareas_a_excluir)].copy()

num_act = tabla_diagnostico.index.str.extract(r'actividad_(\d+)')[0].astype(int)

orden_correcto = (
    pd.DataFrame({'num': num_act, 'nombre': tabla_diagnostico.index})
    .sort_values(by=['num', 'nombre'])['nombre']
    .tolist()
)

tabla_diagnostico = tabla_diagnostico.reindex(orden_correcto)


df_filtrado['supera_90'] = df_filtrado['probabilidad_IA'] >= UMBRAL_ESTRICTO
periodo_control = df_filtrado[df_filtrado['ano'].isin([2021, 2022])]

ruido_dinamico = periodo_control.groupby('actividad_estandar').agg(
    total_entregas=('probabilidad_IA', 'count'),
    falsos_positivos=('supera_90', 'sum')
).reset_index()

ruido_dinamico['porcentaje_ruido'] = (ruido_dinamico['falsos_positivos'] / ruido_dinamico['total_entregas']) * 100

blacklist_actividades = ruido_dinamico[
    ruido_dinamico['porcentaje_ruido'] > TOLERANCIA_MAXIMA_CONTROL
]['actividad_estandar'].unique()

for act in blacklist_actividades:
    print(f" - {act}")

df_clean = df_filtrado[~df_filtrado['actividad_estandar'].isin(blacklist_actividades)].copy()
df_clean['es_IA_estricto'] = df_clean['probabilidad_IA'] >= UMBRAL_ESTRICTO

linea_tiempo_final = df_clean.groupby('ano').agg(
    total_entregas=('probabilidad_IA', 'count'),
    entregas_IA=('es_IA_estricto', 'sum')
).reset_index()

linea_tiempo_final['porcentaje_IA'] = (linea_tiempo_final['entregas_IA'] / linea_tiempo_final['total_entregas']) * 100

print(linea_tiempo_final.to_string(index=False))


plt.figure(figsize=(10, 6))
sns.set_theme(style="whitegrid")

ax = sns.lineplot(data=linea_tiempo_final, x='ano', y='porcentaje_IA', marker='o', linewidth=2.5, color='#1f77b4')

plt.title(f'Evolución Histórica de IA en Entregas (Línea Base Calibrada $\geq$ {UMBRAL_ESTRICTO}%)', fontsize=13, fontweight='bold', pad=15)
plt.xlabel('Año Académico', fontsize=11)
plt.ylabel('Porcentaje de Entregas con IA (%)', fontsize=11)
plt.xticks(linea_tiempo_final['ano'].unique())

for index, fila in linea_tiempo_final.iterrows():
    ax.text(fila['ano'], fila['porcentaje_IA'] + 1.5, f"{fila['porcentaje_IA']:.1f}%", ha="center", fontweight='semibold')

if 2022 in linea_tiempo_final['ano'].values:
    plt.axvline(x=2022.9, color='crimson', linestyle='--', alpha=0.7)

plt.ylim(0, max(linea_tiempo_final['porcentaje_IA'].max() + 15, 40))
plt.tight_layout()
plt.show()


## Análisis de código por tarea específica

Se realiza un análisis del código de una tarea específica

In [ ]:
import pandas as pd
import glob
import os
from collections import Counter

ruta_carpeta = '/content/drive/MyDrive/estudio/UNAL/new_project/data/20260212/submissions/edunext/code/edunext_a10_ejemplo_2_code'

patron_busqueda = os.path.join(ruta_carpeta, "*.py")
archivos_tarea = glob.glob(patron_busqueda)


if len(archivos_tarea) == 0:
    print(" Revisar ruta")
else:
    lineas_globales = []
    longitudes_caracteres = []
    longitudes_lineas = []

    for ruta_archivo in archivos_tarea:
        try:
            with open(ruta_archivo, 'r', encoding='utf-8', errors='ignore') as f:
                contenido = f.read()
                longitudes_caracteres.append(len(contenido))

                lineas = [linea.strip() for linea in contenido.split('\n') if linea.strip()]
                longitudes_lineas.append(len(lineas))
                lineas_globales.extend(lineas)
        except Exception as e:
            continue

    conteo_lineas = Counter(lineas_globales)
    lineas_mas_comunes = conteo_lineas.most_common(10)

    print("=" * 60)
    print(f"Promedio de líneas por entrega: {sum(longitudes_lineas)/len(longitudes_lineas):.1f} líneas.")
    print(f"Promedio de caracteres por entrega: {sum(longitudes_caracteres)/len(longitudes_caracteres):.1f} caracteres.")
    print(f"Total de líneas útiles analizadas: {len(lineas_globales):,}")
    print("=" * 60)
    print("\n Las líneas de código más repetidas (Boilerplate / Plantilla):")
    print("-" * 60)

    for i, (linea, frec) in enumerate(lineas_mas_comunes, 1):
        porcentaje_repeticion = (frec / len(archivos_tarea)) * 100
        print(f" {i}. [{frec} archivos - {porcentaje_repeticion:.1f}%] -> {linea}")
    print("-" * 60)

## Deduplicación

Intenemos deduplicar por usuario.
Importante tener en cuenta que se agrupó por ano, actividad, y username, y se sacó el promedio de valor probabilidad_IA. Tal vez se podría elegir la ultima entrega.

In [ ]:

columnas_sub = ['archive', 'username', 'platform']

df_merged = df_heatmap.merge(
    sub_file[columnas_sub],
    left_on='archivo',
    right_on='archive',
    how='left'
)
df_merged.drop(columns=['archive'], inplace=True)


tareas_a_excluir = [
    'actividad_10_ejemplo_2_code',
    'actividad_11_tarea_2_code',
    'actividad_17_tarea_3_code',
    'actividad_16_ejemplo_3_code',
    'actividad_12_trozos_en_cadenas_code'
]

df_limpio = df_merged[~df_merged['actividad_estandar'].isin(tareas_a_excluir)].copy()


df_por_estudiante = df_limpio.groupby(
    ['ano', 'actividad_estandar', 'username']
)['probabilidad_IA'].mean().reset_index()


In [ ]:

total_filas = len(df_merged)
nulos_username = df_merged['username'].isna().sum()
pct_nulos = (nulos_username / total_filas) * 100

print("=" * 55)
print(f"Total de registros en df_merged:  {total_filas:,}")
print(f"Registros con username VÁLIDO:    {total_filas - nulos_username:,}")
print(f"Registros con username NULO (NaN): {nulos_username:,} ({pct_nulos:.2f}%)")
print("=" * 55)

if nulos_username == 0:
    print("el 100% de los archivos hizo match con un usuario.")
else:
    print("se encontraron registros sin usuario asignado.")
    print(df_merged[df_merged['username'].isna()][['archivo', 'actividad', 'ano']].head(10))

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt



tabla_diagnostico = df_por_estudiante.pivot_table(
    index='actividad_estandar',
    columns='ano',
    values='probabilidad_IA',
    aggfunc='mean'
)

num_act = tabla_diagnostico.index.str.extract(r'actividad_(\d+)')[0].astype(int)

orden_correcto = (
    pd.DataFrame({'num': num_act, 'nombre': tabla_diagnostico.index})
    .sort_values(by=['num', 'nombre'])['nombre']
    .tolist()
)

tabla_diagnostico = tabla_diagnostico.reindex(orden_correcto)

pd.set_option('display.float_format', lambda x: '%.2f' % x if not pd.isna(x) else '-')
print(tabla_diagnostico)


plt.figure(figsize=(14, 10))

sns.heatmap(
    tabla_diagnostico,
    annot=True,
    fmt=".1f",
    cmap="YlOrRd",
    linewidths=.5,
    cbar_kws={'label': 'Probabilidad Promedio de IA (%) [Nivel Estudiante]'}
)

plt.title('Matriz de Diagnóstico: Evolución Pedagógica de IA por Tarea (Actividades 1 a 17)', fontsize=14, pad=15, fontweight='bold')
plt.xlabel('Año Académico', fontsize=12)
plt.ylabel('Nombre de la Actividad / Tarea', fontsize=12)
plt.tight_layout()

plt.savefig('/content/drive/MyDrive/matriz_diagnostico_tareas_ordenada_1_17.png', dpi=300)
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


df_por_estudiante_4anos = df_por_estudiante[df_por_estudiante['ano'] >= 2022].copy()

tabla_diagnostico = df_por_estudiante_4anos.pivot_table(
    index='actividad_estandar',
    columns='ano',
    values='probabilidad_IA',
    aggfunc='mean'
)

num_act = tabla_diagnostico.index.str.extract(r'actividad_(\d+)')[0].astype(int)

orden_correcto = (
    pd.DataFrame({'num': num_act, 'nombre': tabla_diagnostico.index})
    .sort_values(by=['num', 'nombre'])['nombre']
    .tolist()
)

tabla_diagnostico = tabla_diagnostico.reindex(orden_correcto)

pd.set_option('display.float_format', lambda x: '%.2f' % x if not pd.isna(x) else '-')
print(tabla_diagnostico)

plt.figure(figsize=(12, 10))

sns.heatmap(
    tabla_diagnostico,
    annot=True,
    fmt=".1f",
    cmap="YlOrRd",
    linewidths=.5,
    cbar_kws={'label': 'Probabilidad Promedio de IA (%) [Nivel Estudiante]'}
)

plt.title('Matriz de Diagnóstico: Evolución Pedagógica de IA por Tarea (2022 - 2025)', fontsize=14, pad=15, fontweight='bold')
plt.xlabel('Año Académico', fontsize=12)
plt.ylabel('Nombre de la Actividad / Tarea', fontsize=12)
plt.tight_layout()

plt.savefig('/content/drive/MyDrive/matriz_diagnostico_tareas_ordenada_2022_2025.png', dpi=300)
plt.show()

#Diferencias significativas

## Prueba de Kruskal-Wallis

In [ ]:
df_estudiante_4anos = df_por_estudiante[df_por_estudiante['ano'] >= 2022].copy()

In [ ]:
!pip install scikit-posthocs

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import kruskal
import scikit_posthocs as sp

def analizar_kruskal_4anos(df_estudiante, col_tarea='actividad_estandar', col_ano='ano', col_prob='probabilidad_IA'):
    resumen_estadistico = []

    df_temp = df_estudiante.copy()
    df_temp['num_act'] = df_temp[col_tarea].str.extract(r'actividad_(\d+)')[0].astype(float)

    tareas_ordenadas = (
        df_temp.dropna(subset=['num_act'])
        .sort_values(by=['num_act', col_tarea])[col_tarea]
        .unique()
    )

    for tarea in tareas_ordenadas:
        df_t = df_estudiante[df_estudiante[col_tarea] == tarea]

        grupos_anos = [grupo[col_prob].values for _, grupo in df_t.groupby(col_ano)]
        anos_disponibles = sorted(df_t[col_ano].unique())

        if len(grupos_anos) >= 2:
            stat_h, p_val = kruskal(*grupos_anos)
            medianas = df_t.groupby(col_ano)[col_prob].median().to_dict()

            registro = {
                'actividad': tarea,
                'n_total_estudiantes': len(df_t),
                'Kruskal_H_Stat': round(stat_h, 2),
                'p_value': p_val,
                'Significativo (p < 0.05)': 'Sí' if p_val < 0.05 else 'No'
            }

            for a in anos_disponibles:
                registro[f'Mediana_{a}'] = round(medianas.get(a, np.nan), 1)

            resumen_estadistico.append(registro)

    df_resumen = pd.DataFrame(resumen_estadistico)
    df_resumen['p_value_fmt'] = df_resumen['p_value'].apply(
        lambda x: "< 0.0001" if x < 0.0001 else f"{x:.4f}"
    )

    return df_resumen

df_kruskal_4anos = analizar_kruskal_4anos(df_estudiante_4anos)

cols_mostrar = ['actividad', 'n_total_estudiantes', 'Kruskal_H_Stat', 'p_value_fmt', 'Significativo (p < 0.05)']
print(df_kruskal_4anos[cols_mostrar].to_string(index=False))

In [ ]:
import pandas as pd
import scikit_posthocs as sp

def evaluar_dunn_todas_las_tareas(df_estudiante, tareas_significativas):

    filas_dunn = []

    for tarea in tareas_significativas:
        df_t = df_estudiante[df_estudiante['actividad_estandar'] == tarea]

        # Prueba de Dunn
        m_dunn = sp.posthoc_dunn(
            df_t,
            val_col='probabilidad_IA',
            group_col='ano',
            p_adjust='bonferroni'
        )

        anos = sorted(df_t['ano'].unique())

        for i in range(len(anos)):
            for j in range(i + 1, len(anos)):
                a1, a2 = anos[i], anos[j]
                if a1 in m_dunn.index and a2 in m_dunn.columns:
                    p_val = m_dunn.loc[a1, a2]
                    filas_dunn.append({
                        'actividad': tarea,
                        'comparacion': f"{a1} vs {a2}",
                        'p_value': p_val,
                        'p_value_fmt': "< 0.0001" if p_val < 0.0001 else f"{p_val:.4f}",
                        'Diferencia_Significativa': 'Sí' if p_val < 0.05 else 'No'
                    })

    return pd.DataFrame(filas_dunn)

tareas_sig = df_kruskal_4anos[df_kruskal_4anos['Significativo (p < 0.05)'] == 'Sí']['actividad'].tolist()

df_reporte_dunn = evaluar_dunn_todas_las_tareas(df_estudiante_4anos, tareas_sig)

df_vs_2022 = df_reporte_dunn[df_reporte_dunn['comparacion'].str.startswith('2022')]

print(df_vs_2022.to_string(index=False))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


data = [
    {"actividad": "1. Saludar", "2022 vs 2023": 1.0000, "2022 vs 2024": 0.0000, "2022 vs 2025": 0.0000},
    {"actividad": "1. Variables", "2022 vs 2023": 0.8221, "2022 vs 2024": 0.0000, "2022 vs 2025": 0.0000},
    {"actividad": "2. Gritar", "2022 vs 2023": 0.5346, "2022 vs 2024": 0.0000, "2022 vs 2025": 0.0000},
    {"actividad": "3. Dividir en grupos", "2022 vs 2023": 1.0000, "2022 vs 2024": 0.0000, "2022 vs 2025": 0.0000},
    {"actividad": "4. Ejemplo 1", "2022 vs 2023": 0.0294, "2022 vs 2024": 0.0000, "2022 vs 2025": 0.0000},
    {"actividad": "5. Tarea 1", "2022 vs 2023": 0.0127, "2022 vs 2024": 0.0000, "2022 vs 2025": 0.0000},
    {"actividad": "6. Cursores y áreas", "2022 vs 2023": 0.2554, "2022 vs 2024": 0.0000, "2022 vs 2025": 0.0000},
    {"actividad": "7. Años bisiestos", "2022 vs 2023": 0.0000, "2022 vs 2024": 0.0000, "2022 vs 2025": 0.0000},
    {"actividad": "8. While", "2022 vs 2023": 0.0010, "2022 vs 2024": 1.0000, "2022 vs 2025": 1.0000},
    {"actividad": "9. For", "2022 vs 2023": 0.0000, "2022 vs 2024": 0.0001, "2022 vs 2025": 0.0125},
    {"actividad": "13. Listas", "2022 vs 2023": 0.0962, "2022 vs 2024": 0.0000, "2022 vs 2025": 0.0000},
    {"actividad": "14. Diccionarios", "2022 vs 2023": 1.0000, "2022 vs 2024": 0.0000, "2022 vs 2025": 0.0000},
    {"actividad": "15. Conjuntos", "2022 vs 2023": 0.5928, "2022 vs 2024": 0.0000, "2022 vs 2025": 0.0000},
]

df_dunn = pd.DataFrame(data).set_index("actividad")


annot_matrix = df_dunn.applymap(lambda v: f"{v:.4f} *" if v < 0.05 else f"{v:.4f}")

plt.figure(figsize=(9, 10))
sns.set_theme(style="white")

ax = sns.heatmap(
    df_dunn,
    annot=annot_matrix,
    fmt="",
    cmap="YlGnBu_r",
    vmax=0.10,
    vmin=0.00,
    cbar_kws={"label": "p-valor (Ajustado por Bonferroni/Dunn)"},
    linewidths=0.5,
    linecolor="gray"
)

##plt.title("Prueba Post-Hoc de Dunn: Línea Base (2022) vs. Años de Adopción (2023–2025)\n(* Indica p-valor < 0.05, diferencia estadísticamente significativa)", fontsize=11, pad=15, fontweight="bold")
plt.xlabel("Comparación Temporal", fontsize=11, labelpad=10, fontweight="bold")
plt.ylabel("Actividades de Programación", fontsize=11, labelpad=10, fontweight="bold")

plt.xticks(rotation=0, fontsize=10)
plt.yticks(rotation=0, fontsize=10)
plt.tight_layout()

nombre_archivo = os.path.join(imagenes_ia_dir, "5_grafica_ia_5.pdf")
plt.savefig(nombre_archivo, format="pdf", bbox_inches="tight", pad_inches=0.1)

plt.show()

El mapa de calor presenta los resultados de la prueba post-hoc de Dunn, evaluando cómo las distribuciones de las entregas de diversas actividades de programación en los años de adopción (2023–2025) difieren estadísticamente de la línea base establecida en 2022. Se observa que durante 2023 la mayoría de las actividades iniciales no muestran cambios significativos (p-valores elevados en tonos claros), con la excepción notable de tareas de mayor complejidad lógica como "Años bisiestos" y bucles. Sin embargo, al llegar a los años 2024 y 2025, el gráfico experimenta una transformación masiva hacia tonos azul oscuro (p < 0.05), lo que evidencia un cambio estructural generalizado y permanente en las entregas tras la masificación de las herramientas de inteligencia artificial.

Es decir, segun la prueba estadística, a pesar de la llegada en el 2022, el 2023 fue de transición, pero el 2024 es el impacto real.

## Análisis de tareas y su código

In [ ]:
import os
import pandas as pd


ACTIVIDAD_OBJETIVO = 'actividad_7_anos_bisisestos_code'

df_actividad = df_limpio[df_limpio['actividad_estandar'] == 'actividad_7_anos_bisisestos_code'].copy()

print(f"📊 Registros totales en DataFrame para '{ACTIVIDAD_OBJETIVO}': {len(df_actividad):,}")


In [ ]:
df_actividad.head(5)

In [ ]:
sub_file.columns

In [ ]:

columnas_sub = ['archive', 'archivo_origen', 'username', 'platform']

df_merged = df_actividad.merge(
    sub_file[columnas_sub],
    left_on='archivo',
    right_on='archive',
    how='left'
)
df_merged.drop(columns=['archive'], inplace=True)
